# 04 - Analise de Negocio com SQL

Objetivo deste notebook: responder perguntas de negocio usando SQL sobre a tabela analitica consolidada.

Nesta etapa, usamos o banco SQLite criado a partir de `data/processed/orders_analytics.csv`.

Tabela principal: `orders_analytics`.

## 1. Importar bibliotecas

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 2. Conectar ao banco SQLite

In [2]:
PROJECT_ROOT = Path("..").resolve()
DATABASE_PATH = PROJECT_ROOT / "data" / "database" / "olist_analytics.sqlite"

connection = sqlite3.connect(DATABASE_PATH)

DATABASE_PATH

WindowsPath('C:/Users/lucas/brazilian-ecommerce-analytics/data/database/olist_analytics.sqlite')

## 3. Criar uma funcao auxiliar

Esta funcao executa uma consulta SQL e devolve o resultado como DataFrame do pandas.

In [3]:
def run_query(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, connection)

## 4. Quais tabelas existem no banco?

In [4]:
run_query("""
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""")

,table_name
0,orders_analytics


## 5. Quantos pedidos existem por status?

Pergunta de negocio: qual e a situacao operacional dos pedidos?

In [5]:
run_query("""
SELECT
    order_status,
    COUNT(*) AS orders
FROM orders_analytics
GROUP BY order_status
ORDER BY orders DESC;
""")

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 6. Evolucao mensal de pedidos entregues

Pergunta de negocio: como o volume e a receita evoluiram ao longo do tempo?

In [6]:
monthly_delivered_orders = run_query("""
SELECT
    order_year_month,
    COUNT(*) AS delivered_orders,
    ROUND(SUM(product_revenue), 2) AS product_revenue
FROM orders_analytics
WHERE order_status = 'delivered'
GROUP BY order_year_month
ORDER BY order_year_month;
""")

monthly_delivered_orders.head(10)

,order_year_month,delivered_orders,product_revenue
0,2016-09,1,134.97
1,2016-10,265,40325.11
2,2016-12,1,10.90
3,2017-01,750,111798.36
4,2017-02,1653,234223.40
5,2017-03,2546,359198.85
6,2017-04,2303,340669.68
7,2017-05,3546,489338.25
8,2017-06,3135,421923.37
9,2017-07,3872,481604.52


## 7. Categorias com maior receita

Pergunta de negocio: quais categorias mais contribuem para faturamento?

In [7]:
top_categories = run_query("""
SELECT
    main_category,
    COUNT(*) AS orders,
    ROUND(SUM(product_revenue), 2) AS product_revenue,
    ROUND(AVG(product_revenue), 2) AS average_order_revenue
FROM orders_analytics
WHERE order_status = 'delivered'
GROUP BY main_category
ORDER BY product_revenue DESC
LIMIT 10;
""")

top_categories

,main_category,orders,product_revenue,average_order_revenue
0,health_beauty,8613,1233813.10,143.25
1,watches_gifts,5478,1167152.18,213.06
2,bed_bath_table,9184,1024243.76,111.52
3,sports_leisure,7493,954525.89,127.39
4,computers_accessories,6512,888593.98,136.45
5,furniture_decor,6188,709574.80,114.67
6,housewares,5680,615325.93,108.33
7,cool_stuff,3530,610373.43,172.91
8,auto,3793,579364.15,152.75
9,toys,3788,472034.34,124.61


## 8. Estados com maior receita

Pergunta de negocio: onde esta concentrado o faturamento?

In [8]:
top_states = run_query("""
SELECT
    customer_state,
    COUNT(*) AS orders,
    ROUND(SUM(product_revenue), 2) AS product_revenue,
    ROUND(AVG(product_revenue), 2) AS average_order_revenue
FROM orders_analytics
WHERE order_status = 'delivered'
GROUP BY customer_state
ORDER BY product_revenue DESC
LIMIT 10;
""")

top_states

,customer_state,orders,product_revenue,average_order_revenue
0,SP,40501,5067633.16,125.12
1,RJ,12350,1759651.13,142.48
2,MG,11354,1552481.83,136.73
3,RS,5345,728897.47,136.37
4,PR,4923,666063.51,135.30
5,SC,3546,507012.13,142.98
6,BA,3256,493584.14,151.59
7,DF,2080,296498.41,142.55
8,GO,1957,282836.70,144.53
9,ES,1995,268643.45,134.66


## 9. Entregas atrasadas impactam a avaliacao?

Pergunta de negocio: pedidos atrasados recebem notas menores?

In [9]:
late_delivery_review = run_query("""
SELECT
    is_late,
    COUNT(*) AS orders,
    ROUND(AVG(review_score), 2) AS average_review_score,
    ROUND(AVG(delay_days), 2) AS average_delay_days
FROM orders_analytics
WHERE order_status = 'delivered'
  AND is_late IS NOT NULL
  AND review_score IS NOT NULL
GROUP BY is_late
ORDER BY is_late;
""")

late_delivery_review

,is_late,orders,average_review_score,average_delay_days
0,0.0,89443,4.29,-13.51
1,1.0,6381,2.27,10.52


## 10. Categorias com maior taxa de atraso

Pergunta de negocio: em quais categorias o atraso e mais frequente?

Filtro usado: categorias com pelo menos 500 pedidos entregues.

In [10]:
late_rate_by_category = run_query("""
SELECT
    main_category,
    COUNT(*) AS delivered_orders,
    ROUND(100.0 * SUM(CASE WHEN is_late = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_rate_pct,
    ROUND(AVG(review_score), 2) AS average_review_score
FROM orders_analytics
WHERE order_status = 'delivered'
  AND main_category IS NOT NULL
  AND main_category <> 'unknown'
GROUP BY main_category
HAVING delivered_orders >= 500
ORDER BY late_rate_pct DESC
LIMIT 10;
""")

late_rate_by_category

,main_category,delivered_orders,late_rate_pct,average_review_score
0,baby,2761,8.11,4.12
1,office_furniture,1251,8.07,3.65
2,electronics,2502,7.67,4.13
3,musical_instruments,608,7.57,4.24
4,health_beauty,8613,7.54,4.23
5,bed_bath_table,9184,7.48,4.01
6,construction_tools_construction,724,7.46,4.15
7,watches_gifts,5478,7.41,4.12
8,auto,3793,7.33,4.15
9,furniture_decor,6188,7.24,4.08


## 11. Tipos de pagamento

Pergunta de negocio: quais meios de pagamento sao mais usados?

In [11]:
payment_types = run_query("""
SELECT
    main_payment_type,
    COUNT(*) AS orders,
    ROUND(SUM(payment_value), 2) AS payment_value,
    ROUND(AVG(max_installments), 2) AS average_installments
FROM orders_analytics
WHERE order_status = 'delivered'
GROUP BY main_payment_type
ORDER BY orders DESC;
""")

payment_types

,main_payment_type,orders,payment_value,average_installments
0,credit_card,72825,12102206.16,3.55
1,boleto,19191,2769932.58,1.00
2,voucher,2977,341951.91,1.15
3,debit_card,1484,208371.12,1.00
4,NaN,1,NaN,NaN


## 12. Tempo medio de entrega por estado

Pergunta de negocio: quais estados apresentam maior tempo medio de entrega?

In [12]:
delivery_time_by_state = run_query("""
SELECT
    customer_state,
    COUNT(*) AS delivered_orders,
    ROUND(AVG(delivery_days), 2) AS average_delivery_days,
    ROUND(AVG(delay_days), 2) AS average_delay_days,
    ROUND(AVG(review_score), 2) AS average_review_score
FROM orders_analytics
WHERE order_status = 'delivered'
  AND delivery_days IS NOT NULL
GROUP BY customer_state
ORDER BY average_delivery_days DESC;
""")

delivery_time_by_state.head(10)

,customer_state,delivered_orders,average_delivery_days,average_delay_days,average_review_score
0,RR,41,29.39,-17.29,3.90
1,AP,67,27.19,-19.69,4.24
2,AM,145,26.43,-19.57,4.24
3,AL,397,24.54,-8.71,3.85
4,PA,946,23.77,-14.07,3.91
5,MA,717,21.57,-9.57,3.83
6,SE,335,21.52,-10.02,3.91
7,CE,1279,21.27,-10.80,3.94
8,AC,80,21.04,-20.73,4.09
9,PB,517,20.43,-13.26,4.08


## 13. Fechar conexao

In [13]:
connection.close()

## 14. Conclusoes para documentar

Depois de executar as consultas, registre as respostas:

1. Qual status concentra a maior parte dos pedidos?
2. Quais categorias lideram em receita?
3. Quais estados concentram faturamento?
4. Pedidos atrasados possuem nota media menor?
5. Quais categorias possuem maior taxa de atraso?
6. Qual tipo de pagamento e mais usado?
7. Quais estados possuem maior tempo medio de entrega?